In [1]:
import warnings
warnings.filterwarnings('ignore')

from herbie import Herbie
from herbie.toolbox import pc
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from modules import helpers, plot_setup, colormaps
import shutil
import gc
import os
os.environ["ECCODES_MESSAGES_QUIET"] = "1"
import xarray as xr
import cartopy.crs as ccrs

import s3fs

In [2]:
ds = xr.open_zarr("https://data.dynamical.org/noaa/mrms/conus-analysis-hourly/latest.zarr")
lat1, lon1, lat2, lon2 = helpers.map_sectors["NYC Metro"]
prates = ds["precipitation_surface"].sel(time=xr.date_range(start="202109010600", end="202109021200", freq="1h"), latitude=slice(lat2,lat1), longitude=slice(lon1,lon2))

In [8]:
cmli0 = plt.get_cmap("radar")(np.linspace(0/70,10/70,2))
cmli1 = plt.get_cmap("radar")(np.linspace(15/70,29/70,8))
cmli2 = plt.get_cmap("radar")(np.linspace(31/70,60/70,30))
cmli = np.vstack([cmli0, cmli1, cmli2])
cmli[0] = [0,0,0,0]
cmli[1:, 3] = np.ones(len(cmli[1:, 3]))

cm_new = mpl.colors.LinearSegmentedColormap.from_list(colors=cmli, name="radar_short")

In [9]:
for time in prates.time.values:
    fig, ax = plot_setup.make_plot("NYC Metro")
    prate = prates.sel(time=time)

    rainrate = ax.pcolormesh(prate.longitude, 
                    prate.latitude, 
                    (prate.values*3600)/25.4, 
                    transform = pc,
                    cmap = cm_new,
                    vmin=0,
                    vmax=4,
                    zorder=11)

    fig, cax = plot_setup.colorbar(fig, ax, rainrate, "Rainfall Rate (in/hr)", np.arange(0,4.5,0.5))

    plotargs = ["Remnants of Hurricane Ida Flooding", 
            "Multi-Radar Multi-Sensor (MRMS) Analysis", 
            prate,
            "1-hr Averaged Rainfall Rate", 
            "NYC Metro",
            "NOAA/NCEP via dynamical.org"]
    
    plot_setup.finish_plot(fig, ax, plotargs, "group1")

    plt.close(fig)


plot_setup.animate("ida", dir="group1", dur=3)

Done with 06:00Z 01 Sep 2021! Saving...
Done with 07:00Z 01 Sep 2021! Saving...
Done with 08:00Z 01 Sep 2021! Saving...
Done with 09:00Z 01 Sep 2021! Saving...
Done with 10:00Z 01 Sep 2021! Saving...
Done with 11:00Z 01 Sep 2021! Saving...
Done with 12:00Z 01 Sep 2021! Saving...
Done with 13:00Z 01 Sep 2021! Saving...
Done with 14:00Z 01 Sep 2021! Saving...
Done with 15:00Z 01 Sep 2021! Saving...
Done with 16:00Z 01 Sep 2021! Saving...
Done with 17:00Z 01 Sep 2021! Saving...
Done with 18:00Z 01 Sep 2021! Saving...
Done with 19:00Z 01 Sep 2021! Saving...
Done with 20:00Z 01 Sep 2021! Saving...
Done with 21:00Z 01 Sep 2021! Saving...
Done with 22:00Z 01 Sep 2021! Saving...
Done with 23:00Z 01 Sep 2021! Saving...
Done with 00:00Z 02 Sep 2021! Saving...
Done with 01:00Z 02 Sep 2021! Saving...
Done with 02:00Z 02 Sep 2021! Saving...
Done with 03:00Z 02 Sep 2021! Saving...
Done with 04:00Z 02 Sep 2021! Saving...
Done with 05:00Z 02 Sep 2021! Saving...
Done with 06:00Z 02 Sep 2021! Saving...


MoviePy - Done !
MoviePy - video ready loops/ida.mp4
MP4 created and saved at loops/ida.mp4


In [10]:
plot_setup.animate("ida", dir="group1", dur=4)

MoviePy - Building video loops/ida.mp4.
MoviePy - Writing video loops/ida.mp4



MoviePy - Done !
MoviePy - video ready loops/ida.mp4
MP4 created and saved at loops/ida.mp4


In [ ]:
aws = s3fs.S3FileSystem(anon=True)
datestring = "20260318"
data_files, ptype_files = helpers.get_radar_files(datestring)

for i in range(0, 10, 10):
    data, precip_type = helpers.get_radar_data(data_files, ptype_files, i)

    dbz = data.where(data.values > 6.)  
    dbz["longitude"] = dbz.longitude - 360
    precip_type["longitude"] = precip_type.longitude - 360

    fig, ax = plot_setup.make_plot("Special")

    rainrate = ax.pcolormesh(dbz.longitude, 
                    dbz.latitude, 
                    dbz.values, 
                    transform = pc,
                    cmap = "snow", 
                    vmin=0,
                    vmax=60,
                    zorder=11)


    fig, cax = plot_setup.colorbar(fig, ax, rainrate, "Reflectivity (dBZ)", np.arange(0,70,10))

    plotargs = ["Polar Low ... Not a TC!", 
            "Multi-Radar Multi-Sensor (MRMS) Analysis", 
            dbz,
            "Reflectivity at Lowest Altitude (Snow Colormap)", 
            "Southern Alaska",
            "NOAA/NCEP via AWS"]
    
    plot_setup.finish_plot(fig, ax, plotargs, "group1")